In [1]:
# Load in merged data frame 
# copied and pasted from Data/Data Scripts/merge_station_and_event_data_script.py

import pandas as pd
import numpy as np
import os

# What this does: Counts the frequency of non nan's for each feature for each station
# and organizes this information into Features_No_NAN_Counts.csv.

# Note: In the documentation (isd-format-document.pdf) sometimes a number 
# stands in for a nan value. We will deal with that if we choose such a 
# feature later on.


dirc = 'Data/OK City Station Data/Raw Data'
raw_station_data = os.listdir('Data/OK City Station Data/Raw Data') # list files in directory

station_csv_files = [file for file in raw_station_data if ('.csv' in file) and ('Station' in file)] # get csv files

station_list = [] # to hold station numbers as keys and number of csv files as entries

for file in station_csv_files:
    underscore_split = file.split('_')
    station_num = underscore_split[1].replace('.csv','')  
    station_list.append(int(station_num)) # station_num is a string
    


# Read in CSV's as pd.DataFrame's concat along axis = 0 

station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
stations_df = pd.concat(station_pd_dfs, axis = 0)
new_df = stations_df.reset_index().drop(['index', 'Unnamed: 0'], axis =1).copy()



new_df['YEAR-MONTH-DAY'] = new_df['DATE'].apply(lambda r: r.split('T')[0])
new_df['TIME'] = new_df['DATE'].apply(lambda r: r.split('T')[1])
new_df.drop(['DATE'],axis=1,inplace=True)

# combined oklahoma tornadoes

path = 'Data/Storm Event Data/Cleaned Data/oklahoma_tornadoes_2000_2021.csv'
df2 = pd.read_csv(path).copy()
# Really will only care about 'BEGIN_DATE_TIME', 'END_DATE_TIME', 'BEGIN_LAT',
# 'END_LAT', 'BEGIN_LON', 'END_LON'

df2 = df2[
        [
        'BEGIN_DATE_TIME', 
        'END_DATE_TIME', 
        'BEGIN_LAT', 
        'END_LAT', 
        'BEGIN_LON', 
        'END_LON'
        ]
        ]

# columns BEGIN-DATE and BEGIN-TIME
new_df2 = df2.copy()
new_df2['BEGIN_DATE'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['BEGIN_TIME'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[1])
new_df2['END_DATE'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['END_TIME'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[1])

new_df2.drop(['BEGIN_DATE_TIME','END_DATE_TIME'],axis=1,inplace=True)

# Change format of 'BEGIN_DATE' and 'END_DATE' to match that of 'YEAR_MONTH_DAY' in station data

month_num = {
            'JAN':'01',
            'FEB':'02',
            'MAR':'03', 
            'APR':'04', 
            'MAY':'05', 
            'JUN':'06', 
            'JUL':'07', 
            'AUG':'08', 
            'SEP':'09', 
            'OCT':'10', 
            'NOV':'11', 
            'DEC':'12'
            }

new_df2['BEGIN_DATE'] = new_df2['BEGIN_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')
new_df2['END_DATE'] = new_df2['END_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')

# Get year to be 20**
new_df2['BEGIN_DATE']=new_df2['BEGIN_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')
new_df2['END_DATE']=new_df2['END_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')

final_df = pd.merge(left=new_df,right=new_df2,how ='outer',left_on='YEAR-MONTH-DAY',right_on='BEGIN_DATE')

/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14677/1313703029.py:32: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14677/1313703029.py:32: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_14677/1313703029.py:32: DtypeWarning: Columns (15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,32,34,40,41,4

In [2]:
final_df

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,SOURCE,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,AA1,...,YEAR-MONTH-DAY,TIME,BEGIN_LAT,END_LAT,BEGIN_LON,END_LON,BEGIN_DATE,BEGIN_TIME,END_DATE,END_TIME
0,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,SY-MT,OKC,V020,"01,0000,9,5",...,2000-01-01,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1375491,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,19:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1375492,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,20:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1375493,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,21:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1375494,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,22:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
